# React 框架 · 第 09 天

## 极客头条：文章管理、编辑与构建优化

**资料范围**：day09 课件 12 页、配套项目总笔记中“文章列表 / 编辑文章 / 项目打包”部分，以及 `react-jike` 随堂代码。虽然 day06～day09 压缩包都附带同一套项目总笔记，day09 的课件明确聚焦本篇主题，因此这里不重复登录和 Layout 的入门内容。

**学习主线**：筛选参数 → 列表请求 → 分页/删除 → 编辑回填 → 提交更新 → 打包与优化。

## 一、学习目标与整体数据流

完成本日后，应能：

1. 将筛选表单转换为文章列表接口的参数，并在参数变化时刷新表格。
2. 让 Ant Design `Table` 的分页状态和服务端分页参数保持一致。
3. 删除文章后正确刷新列表，处理删除当前页最后一条的边界。
4. 通过文章 ID 进入编辑页，回填普通字段和封面图片。
5. 让同一张表单兼容新增与编辑，正确转换新上传/旧图片 URL。
6. 构建、预览应用，并理解路由懒加载、包体积分析和 CDN 的适用条件。

```text
Form 筛选值 ──转换──> 请求参数 params ──请求──> 文章列表 ──渲染──> Table
Table 翻页 ──────────┘                     │
删除成功 ──刷新当前列表────────────────────┘
编辑按钮 ──文章 id──> Publish 页 ──详情回填──> 修改并提交 PUT
```

## 二、先认识文章接口与返回结构

随堂代码把文章相关请求集中在 `src/apis/article.js`：

```js
export const getArticleListAPI = params => request({
  url: '/mp/articles', method: 'GET', params,
})

export const delArticleAPI = id => request({
  url: `/mp/articles/${id}`, method: 'DELETE',
})

export const getArticleById = id => request({
  url: `/mp/articles/${id}`, method: 'GET',
})

export const createArticleAPI = data => request({
  url: '/mp/articles?draft=false', method: 'POST', data,
})

export const updateArticleAPI = (id, data) => request({
  url: `/mp/articles/${id}?draft=false`, method: 'PUT', data,
})
```

课程的 `request` 响应拦截器返回 `response.data`，所以业务层拿到的 `res` 已经是接口响应体；示例列表数据通过 `res.data.results`、`res.data.total_count` 读取。封装返回值时应始终核对实际接口结构，避免多取或少取一层 `data`。

## 三、列表筛选：表单值转换为请求参数

筛选本质是：**给同一个列表接口传不同的参数，取得不同结果**。本项目的参数包含状态、频道、起止发布日期以及分页字段。

```jsx
const [params, setParams] = useState({
  status: '',
  channel_id: '',
  begin_pubdate: '',
  end_pubdate: '',
  page: 1,
  per_page: 4,
})

const onFinish = values => {
  const { status, channel_id, date } = values
  setParams(prev => ({
    ...prev,
    status: status ?? '',
    channel_id: channel_id ?? '',
    begin_pubdate: date?.[0]?.format('YYYY-MM-DD') ?? '',
    end_pubdate: date?.[1]?.format('YYYY-MM-DD') ?? '',
    page: 1, // 新筛选条件从第一页重新查
  }))
}
```

课件给出的步骤依次是准备完整参数、获取表单值、按接口字段转换、重新请求列表。**日期范围是可选项**，直接写 `date[0]` 会在未选日期时抛错，因此要做空值处理。筛选条件变化时重置 `page`，否则可能停留在旧条件下的后几页，出现“有结果却显示空页”。

## 四、参数变化驱动列表请求

```jsx
const [article, setArticle] = useState({ list: [], count: 0 })
const [loading, setLoading] = useState(false)
const [error, setError] = useState('')

useEffect(() => {
  let cancelled = false

  async function loadArticles() {
    setLoading(true)
    setError('')
    try {
      const res = await getArticleListAPI(params)
      if (!cancelled) {
        setArticle({
          list: res.data.results ?? [],
          count: res.data.total_count ?? 0,
        })
      }
    } catch (err) {
      if (!cancelled) setError('文章列表加载失败')
    } finally {
      if (!cancelled) setLoading(false)
    }
  }

  loadArticles()
  return () => { cancelled = true }
}, [params])
```

`params` 变化后，Effect 重新调用接口并更新 `Table`。`cancelled` 只阻止旧请求回写界面；如果需要真正取消网络请求，可进一步使用请求工具支持的取消机制。加载和错误状态让用户知道请求进展，避免空表格看起来像“没有数据”。

## 五、服务端分页与 `Table` 同步

Ant Design Table 的分页器需要总数、当前页、每页条数和切换事件：

```jsx
<Table
  rowKey="id"
  columns={columns}
  dataSource={article.list}
  loading={loading}
  pagination={{
    current: params.page,
    pageSize: params.per_page,
    total: article.count,
    onChange: page => {
      setParams(prev => ({ ...prev, page }))
    },
  }}
/>
```

**区分两种分页**：如果接口已按 `page` 和 `per_page` 返回当前页，Table 只负责显示分页器，不再对已返回的数组二次切片。总数来自接口的 `total_count`，不是当前页的 `list.length`。

![文章列表与分页示意](assets/day09/21.png)

## 六、删除文章：确认、请求和刷新

删除按钮放在表格的“操作”列中，先通过 `Popconfirm` 让用户确认，再用当前行的 `id` 调接口：

```jsx
const [reloadKey, setReloadKey] = useState(0)

const onDelete = async record => {
  await delArticleAPI(record.id)
  message.success('删除成功')

  // 当前页删到空页时，回到上一页；否则重新请求当前页。
  if (article.list.length === 1 && params.page > 1) {
    setParams(prev => ({ ...prev, page: prev.page - 1 }))
  } else {
    setReloadKey(value => value + 1)
  }
}
```

```jsx
<Popconfirm
  title="删除文章"
  description="确认删除当前文章吗？"
  okText="确认"
  cancelText="取消"
  onConfirm={() => onDelete(record)}
>
  <Button danger icon={<DeleteOutlined />} />
</Popconfirm>
```

若采用 `reloadKey`，列表请求 Effect 的依赖应改为 `[params, reloadKey]`。课程代码通过 `setReqData({ ...reqData })` 触发重查，原理是创建新对象；显式的刷新键更容易看懂。删除失败时应保留原列表并提示错误。

## 七、编辑按钮把文章 ID 带到发布页

课程复用 `/publish` 页面：没有 id 时发布，有 id 时编辑。

```jsx
const navigate = useNavigate()

<Button
  type="primary"
  icon={<EditOutlined />}
  onClick={() => navigate(`/publish?id=${record.id}`)}
/>
```

发布页读取查询参数：

```jsx
const [searchParams] = useSearchParams()
const articleId = searchParams.get('id')
const isEdit = Boolean(articleId)
```

ID 是文章的身份标识，后续详情请求与更新请求都使用同一个 ID。路径参数 `/publish/:id` 也可以实现，但应和路由表、跳转代码、详情页读取方式保持一致。

![编辑入口示意](assets/day09/22.png)

## 八、编辑页回填普通表单字段

Ant Design Form 的实例由 `Form.useForm()` 提供。详情请求成功后，使用 `setFieldsValue` 回填异步数据：

```jsx
const [form] = Form.useForm()

useEffect(() => {
  if (!articleId) return
  let cancelled = false

  async function loadDetail() {
    const res = await getArticleById(articleId)
    if (cancelled) return
    const { cover, ...fields } = res.data
    form.setFieldsValue({
      ...fields,
      type: cover?.type ?? 0,
    })
  }

  loadDetail()
  return () => { cancelled = true }
}, [articleId, form])

return <Form form={form} onFinish={onFinish}>{/* 表单项 */}</Form>
```

`initialValues` 只适合初始化表单，异步获取详情后要使用 `setFieldsValue`。文章标题、频道、内容等普通字段直接回填；封面图片还要转换成 Upload 需要的结构。

## 九、回填封面类型与图片列表

接口的封面数据形如：

```js
cover: { type: 3, images: ['url-a', 'url-b', 'url-c'] }
```

`Radio.Group` 需要封面类型，`Upload` 的受控 `fileList` 需要文件对象数组：

```jsx
const [imageType, setImageType] = useState(0)
const [imageList, setImageList] = useState([])

// 详情请求成功后
setImageType(cover?.type ?? 0)
setImageList((cover?.images ?? []).map((url, index) => ({
  uid: `saved-${index}`,
  name: `cover-${index + 1}`,
  status: 'done',
  url,
})))
```

```jsx
<Form.Item name="type" label="封面类型">
  <Radio.Group onChange={event => setImageType(event.target.value)}>
    <Radio value={0}>无图</Radio>
    <Radio value={1}>单图</Radio>
    <Radio value={3}>三图</Radio>
  </Radio.Group>
</Form.Item>

{imageType > 0 && (
  <Upload fileList={imageList} onChange={info => setImageList(info.fileList)} />
)}
```

`fileList` 和 `onChange` 配合，让 Upload 成为受控组件。回填旧图片时，关键是把 URL 包装成带 `url` 的文件项，才能在预览列表中显示。

![编辑封面回填示意](assets/day09/24.png)

## 十、同一张表单兼容“发布”和“编辑”

根据是否有 `articleId` 调整标题、按钮文案和提交接口：

```jsx
<Card title={isEdit ? '编辑文章' : '发布文章'}>
  <Form form={form} onFinish={onFinish}>
    {/* 标题、频道、内容、封面字段 */}
    <Button type="primary" htmlType="submit">
      {isEdit ? '更新文章' : '发布文章'}
    </Button>
  </Form>
</Card>
```

这样可以复用表单结构和校验规则，但要明确两种模式的差别：

| 项目 | 发布 | 编辑 |
| --- | --- | --- |
| URL 是否有 id | 否 | 是 |
| 是否请求详情 | 否 | 是 |
| 是否回填表单 | 否 | 是 |
| 提交方法 | `POST` | `PUT` |
| 成功提示 | 发布成功 | 更新成功 |

## 十一、提交更新：处理新旧图片的两种格式

新上传图片的地址通常在 `file.response.data.url`，回填图片的地址在 `file.url`。提交前必须统一转换：

```jsx
const toCoverUrl = file => file.response?.data?.url ?? file.url

const onFinish = async values => {
  if (imageType !== imageList.length) {
    message.warning('封面类型和图片数量不匹配')
    return
  }

  const images = imageList.map(toCoverUrl)
  if (images.some(url => !url)) {
    message.warning('图片仍在上传或上传失败')
    return
  }

  const data = {
    title: values.title,
    content: values.content,
    channel_id: values.channel_id,
    cover: { type: imageType, images },
  }

  try {
    if (isEdit) {
      await updateArticleAPI(articleId, data)
    } else {
      await createArticleAPI(data)
    }
    message.success(isEdit ? '更新成功' : '发布成功')
    navigate('/article')
  } catch (error) {
    message.error(isEdit ? '更新失败' : '发布失败')
  }
}
```

随堂代码中的 `updateArticleAPI({ ...reqData, id })` 也可工作，但更清晰的接口设计是把 `id` 与请求体分开，避免将 id 无意间发送进 body。提交前应等待异步请求完成，再提示成功和跳转。

## 十二、打包与本地预览

开发完成后运行：

```bash
npm run build
```

构建产物在课程 CRA 项目的 `build/` 中。可用静态服务模拟部署环境：

```bash
npx serve -s build
```

课程材料使用全局安装 `serve` 再运行 `serve -s ./build`；这里用 `npx` 省去全局安装步骤，作用相同。预览时检查登录、刷新深层路由、图片资源、文章列表和编辑页是否正常。`createBrowserRouter` 使用 history 地址，真实服务器也要把前端路径回退到入口 HTML。

![构建与预览流程](assets/day09/25.png)

## 十三、路由懒加载

路由懒加载让某个页面的 JS 在用户访问该路由时再获取，从而缩小初始加载量。随堂代码使用 `lazy` 和 `Suspense`：

```jsx
import { lazy, Suspense } from 'react'

const Home = lazy(() => import('@/pages/Home'))
const Article = lazy(() => import('@/pages/Article'))
const Publish = lazy(() => import('@/pages/Publish'))

const withFallback = Component => (
  <Suspense fallback={<div>加载中...</div>}>
    <Component />
  </Suspense>
)

const router = createBrowserRouter([
  {
    path: '/',
    element: <AuthRoute><Layout /></AuthRoute>,
    children: [
      { index: true, element: withFallback(Home) },
      { path: 'article', element: withFallback(Article) },
      { path: 'publish', element: withFallback(Publish) },
    ],
  },
])
```

实际效果可以在构建后打开浏览器 Network：首次访问时只加载必要资源，进入文章列表或发布页时再加载对应代码块。懒加载减少首屏请求体积，同时也增加了路由切换时的动态请求，所以要提供可见的加载状态。

## 十四、分析构建体积

使用 `source-map-explorer` 可以把打包后的文件按依赖来源可视化，找出体积较大的模块：

```bash
npm i -D source-map-explorer
```

```json
{
  "scripts": {
    "analyze": "source-map-explorer 'build/static/js/*.js'"
  }
}
```

```bash
npm run build
npm run analyze
```

分析时重点看：哪些第三方库占据较大比例、是否把只在某页用到的库带进了首屏、路由拆分是否生效。体积分析是寻找优化目标的证据，不应仅凭包名直觉删除依赖。

![包体积分析示意](assets/day09/27.png)

## 十五、CDN 优化的工作方式与核查

课程提出把 React、ReactDOM 等稳定的大型第三方库从应用 bundle 中排除，再通过 CDN 脚本加载：

```text
webpack externals: react → window.React, react-dom → window.ReactDOM
HTML: 先加载对应 CDN 脚本，再加载应用 bundle
```

在 CRACO 的生产构建配置中指定 `externals`，并通过 HTML 模板注入 `<script>`。这一优化成立需要同时满足：

1. CDN 脚本版本与项目依赖兼容；课程代码的 CDN 示例是 18.1.0，而配套 `package.json` 中 React 为 18.2.0，照抄前应核对版本。
2. 脚本加载顺序正确，应用代码执行前 `React` 与 `ReactDOM` 已存在。
3. CDN 不可用时有回退策略，否则整个应用可能无法启动。
4. 实测首屏耗时和缓存命中收益，不只比较打包文件大小。

CDN 是可选的部署优化；路由懒加载和移除无用依赖通常更容易验证。

## 十六、容易踩的坑

- **未选日期就点击筛选**：`date` 可能为空；使用可选链，并按接口约定填空值或省略字段。
- **切换筛选条件后列表为空**：可能还停留在旧条件的第 N 页；筛选时重置 `page: 1`。
- **表格总页数不对**：分页 `total` 应使用接口的 `total_count`。
- **删除后没有刷新**：删除成功后显式触发重查；删除最后一条时考虑回到上一页。
- **编辑时普通字段回填了，封面却不显示**：Upload 要的是 `fileList`，不能直接传 URL 字符串数组。
- **编辑提交时封面 URL 为 undefined**：新上传图片读 `response.data.url`，旧图读 `url`。
- **更新失败但提示成功**：对接口调用 `await`，只在成功分支展示成功提示。
- **懒加载后页面一瞬间空白**：用 `Suspense fallback` 提供加载提示。
- **CDN 后页面空白**：检查脚本 URL、版本、加载顺序和 `externals` 全局变量名。

## 十七、练习与验收清单

建议按以下顺序自测：

1. 不选日期、只选状态时能正常筛选。
2. 切换频道、日期后自动回到第一页，表格和总数一致。
3. 翻页后 URL 不变也能请求对应页，刷新列表不会丢掉筛选条件。
4. 删除普通行后当前页更新；删除当前页最后一行后回到合理页。
5. 点击编辑进入 `/publish?id=...`，标题、频道、内容和封面都正确回填。
6. 编辑时保留旧图、替换新图、无图/单图/三图切换后均能形成正确请求体。
7. 构建成功，本地预览可访问登录页、文章列表和编辑页。
8. Network 中可观察到路由代码块按需加载，体积分析结果与预期一致。

这些检查覆盖了“表单 → 请求 → 状态 → UI → 构建”的完整链路。

## 十八、本日总结

第 09 天的核心是把文章管理闭环做完：筛选参数决定列表请求；分页和删除持续维护当前列表；文章 ID 连接列表与编辑页；编辑页通过详情接口回填表单和封面，再复用发布表单提交更新。最后通过构建、本地预览、路由懒加载和包体积分析确认项目可以交付并具备基本性能意识。

**记忆主线**：`Form values → params → GET list → Table → id → GET detail → setFieldsValue + fileList → PUT → build`。